# Debug bmcs_utils Import Issues

This notebook investigates hanging import issues with the bmcs_utils package.
We'll test imports systematically to identify bottlenecks and potential circular dependencies.

In [1]:
import sys
import time
import importlib
import threading
from contextlib import contextmanager

# Setup timing decorator
@contextmanager
def timer(description):
    start = time.time()
    print(f"Starting: {description}")
    try:
        yield
    finally:
        end = time.time()
        print(f"Completed: {description} in {end - start:.3f} seconds")

# Function to test import with timeout
def import_with_timeout(module_name, timeout=10):
    """Import module with timeout to detect hanging imports"""
    result = {"success": False, "error": None, "module": None}
    
    def import_target():
        try:
            result["module"] = importlib.import_module(module_name)
            result["success"] = True
        except Exception as e:
            result["error"] = str(e)
    
    thread = threading.Thread(target=import_target)
    thread.daemon = True
    thread.start()
    thread.join(timeout)
    
    if thread.is_alive():
        print(f"WARNING: Import of {module_name} is hanging (>{timeout}s)")
        return None
    elif result["success"]:
        print(f"SUCCESS: {module_name} imported successfully")
        return result["module"]
    else:
        print(f"ERROR: {module_name} failed - {result['error']}")
        return None

print("Import testing utilities ready")

Import testing utilities ready


## Test 1: Individual Module Imports

Test each component of bmcs_utils individually to identify problematic modules.

In [2]:
# Test individual modules that are imported in api.py
modules_to_test = [
    'bmcs_utils',
    'bmcs_utils.app_window',
    'bmcs_utils.model',
    'bmcs_utils.model_list',
    'bmcs_utils.model_dict',
    'bmcs_utils.view',
    'bmcs_utils.item',
    'bmcs_utils.mpl_utils',
    'bmcs_utils.symb_expr',
    'bmcs_utils.trait_types',
    'bmcs_utils.editors',
    'bmcs_utils.misc.plot_tools',
    'bmcs_utils.parametric_study',
    'bmcs_utils.data_cache',
    'bmcs_utils.symbol.cymbol'
]

results = {}
for module in modules_to_test:
    with timer(f"Import {module}"):
        results[module] = import_with_timeout(module, timeout=15)
        
print("\n=== IMPORT SUMMARY ===")
for module, result in results.items():
    status = "✓" if result is not None else "✗"
    print(f"{status} {module}")

Starting: Import bmcs_utils
SUCCESS: bmcs_utils imported successfully
Completed: Import bmcs_utils in 0.003 seconds
Starting: Import bmcs_utils.app_window
SUCCESS: bmcs_utils.app_window imported successfully
Completed: Import bmcs_utils.app_window in 0.209 seconds
Starting: Import bmcs_utils.model
SUCCESS: bmcs_utils.model imported successfully
Completed: Import bmcs_utils.model in 0.022 seconds
Starting: Import bmcs_utils.model_list
SUCCESS: bmcs_utils.model_list imported successfully
Completed: Import bmcs_utils.model_list in 0.007 seconds
Starting: Import bmcs_utils.model_dict
SUCCESS: bmcs_utils.model_dict imported successfully
Completed: Import bmcs_utils.model_dict in 0.003 seconds
Starting: Import bmcs_utils.view
SUCCESS: bmcs_utils.view imported successfully
Completed: Import bmcs_utils.view in 0.000 seconds
Starting: Import bmcs_utils.item
SUCCESS: bmcs_utils.item imported successfully
Completed: Import bmcs_utils.item in 0.000 seconds
Starting: Import bmcs_utils.mpl_utils
SUC

## Test 2: Check for Circular Dependencies

Analyze the import graph to detect potential circular imports.

In [3]:
import ast
import os
from pathlib import Path

def find_imports_in_file(filepath):
    """Extract all imports from a Python file"""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
        
        tree = ast.parse(content)
        imports = []
        
        for node in ast.walk(tree):
            if isinstance(node, ast.Import):
                for alias in node.names:
                    imports.append(alias.name)
            elif isinstance(node, ast.ImportFrom):
                if node.module:
                    imports.append(node.module)
                    
        return imports
    except Exception as e:
        print(f"Error parsing {filepath}: {e}")
        return []

def build_import_graph(package_path):
    """Build import dependency graph for the package"""
    import_graph = {}
    package_root = Path(package_path)
    
    for py_file in package_root.rglob("*.py"):
        if py_file.name == "__init__.py":
            continue
            
        # Convert file path to module name
        rel_path = py_file.relative_to(package_root.parent)
        module_name = str(rel_path.with_suffix('')).replace('/', '.')
        
        imports = find_imports_in_file(py_file)
        # Filter only bmcs_utils imports
        bmcs_imports = [imp for imp in imports if imp.startswith('bmcs_utils')]
        import_graph[module_name] = bmcs_imports
        
    return import_graph

# Build and analyze import graph
bmcs_path = "/home/rch/Coding/bmcs_utils/bmcs_utils"
if os.path.exists(bmcs_path):
    print("Building import dependency graph...")
    import_graph = build_import_graph(bmcs_path)
    
    print("\n=== IMPORT DEPENDENCIES ===")
    for module, deps in import_graph.items():
        if deps:
            print(f"{module}:")
            for dep in deps:
                print(f"  -> {dep}")
else:
    print(f"Package path not found: {bmcs_path}")

Building import dependency graph...

=== IMPORT DEPENDENCIES ===
bmcs_utils.app_window_example:
  -> bmcs_utils.app_window
  -> bmcs_utils.model
  -> bmcs_utils.view
  -> bmcs_utils.item
  -> bmcs_utils.trait_types
bmcs_utils.tree_node:
  -> bmcs_utils.controller
bmcs_utils.symb_expr_quad_example:
  -> bmcs_utils.symb_expr
bmcs_utils.api:
  -> bmcs_utils
  -> bmcs_utils.app_window
  -> bmcs_utils.model
  -> bmcs_utils.model_list
  -> bmcs_utils.model_dict
  -> bmcs_utils.view
  -> bmcs_utils.item
  -> bmcs_utils.mpl_utils
  -> bmcs_utils.symb_expr
  -> bmcs_utils.trait_types
  -> bmcs_utils.editors
  -> bmcs_utils.misc.plot_tools
  -> bmcs_utils.parametric_study
  -> bmcs_utils.data_cache
  -> bmcs_utils.symbol.cymbol
  -> bmcs_utils.k3d_utils.extrusion_for_3d_curve
  -> bmcs_utils.k3d_utils.k3d_utils
bmcs_utils.model:
  -> bmcs_utils.view
  -> bmcs_utils.i_model
  -> bmcs_utils.app_window
bmcs_utils.app_window:
  -> bmcs_utils.i_model
bmcs_utils.view:
  -> bmcs_utils.item
  -> bmcs_ut

## Test 3: Progressive Import Testing

Test importing the main api.py step by step to identify the exact hanging point.

In [4]:
# Clear any previously imported bmcs_utils modules
modules_to_remove = [name for name in sys.modules.keys() if name.startswith('bmcs_utils')]
for module in modules_to_remove:
    del sys.modules[module]
    
print(f"Cleared {len(modules_to_remove)} bmcs_utils modules from cache")

# Test the main api import
print("\n=== TESTING MAIN API IMPORT ===")
with timer("Full bmcs_utils.api import"):
    api_module = import_with_timeout('bmcs_utils.api', timeout=30)

if api_module:
    print("✓ API imported successfully")
    print("Available attributes:")
    attrs = [attr for attr in dir(api_module) if not attr.startswith('_')]
    for attr in sorted(attrs):
        print(f"  - {attr}")
else:
    print("✗ API import failed or hung")

Cleared 41 bmcs_utils modules from cache

=== TESTING MAIN API IMPORT ===
Starting: Full bmcs_utils.api import
SUCCESS: bmcs_utils.api imported successfully
Completed: Full bmcs_utils.api import in 0.775 seconds
✓ API imported successfully
Available attributes:
  - AppWindow
  - Array
  - ArrayEditor
  - Bool
  - BoolEditor
  - Button
  - ButtonEditor
  - Cymbol
  - Dict
  - EitherType
  - EitherTypeEditor
  - Enum
  - Extruder
  - Float
  - FloatEditor
  - FloatRangeEditor
  - FloatSliderEditor
  - FloatSliderEditorSelector
  - HistoryEditor
  - InjectSymbExpr
  - Instance
  - InstanceEditor
  - Int
  - IntEditor
  - IntRangeEditor
  - InteractiveModel
  - InteractiveWindow
  - Item
  - K3DUtils
  - List
  - ListEditor
  - Model
  - ModelDict
  - ModelList
  - ParametricStudy
  - Progress
  - ProgressEditor
  - Range
  - Selector
  - Str
  - SymbExpr
  - TextAreaEditor
  - View
  - WeakRef
  - bmcs_utils
  - ccode
  - cymbols
  - data_cache
  - mpl_align_xaxis
  - mpl_align_yaxis
  - 

## Test 4: Memory and Threading Analysis

Check for potential issues with threading, memory usage, or blocking operations.

In [5]:
import psutil
import gc
import threading

def get_memory_usage():
    """Get current memory usage"""
    process = psutil.Process()
    return process.memory_info().rss / 1024 / 1024  # MB

def get_thread_count():
    """Get current thread count"""
    return threading.active_count()

print("=== SYSTEM STATE BEFORE IMPORT ===")
print(f"Memory usage: {get_memory_usage():.1f} MB")
print(f"Active threads: {get_thread_count()}")
print(f"Objects in memory: {len(gc.get_objects())}")

# Clear modules again for clean test
modules_to_remove = [name for name in sys.modules.keys() if name.startswith('bmcs_utils')]
for module in modules_to_remove:
    del sys.modules[module]
gc.collect()

print("\n=== IMPORTING WITH MONITORING ===")
start_memory = get_memory_usage()
start_threads = get_thread_count()

with timer("Monitored API import"):
    try:
        import bmcs_utils.api
        success = True
    except Exception as e:
        print(f"Import error: {e}")
        success = False

print(f"\n=== SYSTEM STATE AFTER IMPORT ===")
print(f"Import successful: {success}")
print(f"Memory usage: {get_memory_usage():.1f} MB (Δ{get_memory_usage() - start_memory:+.1f} MB)")
print(f"Active threads: {get_thread_count()} (Δ{get_thread_count() - start_threads:+})")
print(f"Objects in memory: {len(gc.get_objects())}")

=== SYSTEM STATE BEFORE IMPORT ===
Memory usage: 246.9 MB
Active threads: 7
Objects in memory: 250489

=== IMPORTING WITH MONITORING ===
Starting: Monitored API import
Completed: Monitored API import in 0.034 seconds

=== SYSTEM STATE AFTER IMPORT ===
Import successful: True
Memory usage: 248.6 MB (Δ+0.4 MB)
Active threads: 7 (Δ+0)
Objects in memory: 252160


## Test 5: Conditional Import Testing

Test the conditional K3D imports and other optional dependencies.

In [6]:
# Test conditional imports that might cause issues
print("=== TESTING CONDITIONAL IMPORTS ===")

# Check bmcs_utils.ENABLE_K3D flag
try:
    import bmcs_utils
    k3d_enabled = bmcs_utils.ENABLE_K3D
    print(f"K3D enabled: {k3d_enabled}")
except:
    print("Could not check K3D flag")
    k3d_enabled = False

if k3d_enabled:
    print("\nTesting K3D imports:")
    with timer("K3D extrusion import"):
        k3d_extruder = import_with_timeout('bmcs_utils.k3d_utils.extrusion_for_3d_curve', timeout=10)
    
    with timer("K3D utils import"):
        k3d_utils = import_with_timeout('bmcs_utils.k3d_utils.k3d_utils', timeout=10)
else:
    print("K3D imports skipped (disabled)")

# Test symbolic computation imports
print("\n=== TESTING SYMBOLIC COMPUTATION ===")
with timer("Cymbol import"):
    cymbol = import_with_timeout('bmcs_utils.symbol.cymbol', timeout=15)

if cymbol:
    print("Testing basic cymbol functionality...")
    try:
        # Test if the symbolic system works without hanging
        cymbols = cymbol.cymbols
        test_symbol = cymbols('x')
        print(f"✓ Created test symbol: {test_symbol}")
    except Exception as e:
        print(f"✗ Cymbol functionality test failed: {e}")

=== TESTING CONDITIONAL IMPORTS ===
K3D enabled: True

Testing K3D imports:
Starting: K3D extrusion import
SUCCESS: bmcs_utils.k3d_utils.extrusion_for_3d_curve imported successfully
Completed: K3D extrusion import in 0.001 seconds
Starting: K3D utils import
SUCCESS: bmcs_utils.k3d_utils.k3d_utils imported successfully
Completed: K3D utils import in 0.000 seconds

=== TESTING SYMBOLIC COMPUTATION ===
Starting: Cymbol import
SUCCESS: bmcs_utils.symbol.cymbol imported successfully
Completed: Cymbol import in 0.000 seconds
Testing basic cymbol functionality...
✓ Created test symbol: (x,)


## Test 6: Import from External Package Simulation

Simulate how an external package would import bmcs_utils to reproduce the hanging issue.

In [7]:
# Simulate external package import scenario
print("=== SIMULATING EXTERNAL PACKAGE IMPORT ===")

# Clear all bmcs_utils modules to simulate fresh import
modules_to_remove = [name for name in sys.modules.keys() if name.startswith('bmcs_utils')]
for module in modules_to_remove:
    del sys.modules[module]

print(f"Cleared {len(modules_to_remove)} modules")

# Simulate what an external package might do
def simulate_external_import():
    """Simulate how external package imports bmcs_utils"""
    import importlib
    
    # Common import patterns that might cause issues
    patterns = [
        "from bmcs_utils.api import *",
        "import bmcs_utils.api",
        "from bmcs_utils import api",
        "import bmcs_utils"
    ]
    
    for pattern in patterns:
        print(f"\nTesting pattern: {pattern}")
        try:
            if pattern.startswith("from") and pattern.endswith("import *"):
                # Handle star imports
                module_path = pattern.split()[1]
                module = importlib.import_module(module_path)
                # Simulate accessing all attributes
                attrs = [attr for attr in dir(module) if not attr.startswith('_')]
                print(f"  Found {len(attrs)} public attributes")
            else:
                # Handle regular imports
                exec(pattern)
                print(f"  ✓ Success")
        except Exception as e:
            print(f"  ✗ Error: {e}")

with timer("External import simulation"):
    simulate_external_import()

=== SIMULATING EXTERNAL PACKAGE IMPORT ===
Cleared 45 modules
Starting: External import simulation

Testing pattern: from bmcs_utils.api import *
  Found 55 public attributes

Testing pattern: import bmcs_utils.api
  ✓ Success

Testing pattern: from bmcs_utils import api
  ✓ Success

Testing pattern: import bmcs_utils
  ✓ Success
Completed: External import simulation in 0.040 seconds


## Test 7: Recommendations and Solutions

Based on the test results, provide recommendations for fixing the hanging import issues.

In [8]:
print("=== ANALYSIS AND RECOMMENDATIONS ===")
print("""
Based on the tests above, common causes of hanging imports include:

1. **Circular Dependencies**: Check if modules import each other
2. **Heavy Initialization**: Long-running code in module-level scope
3. **Threading Issues**: Deadlocks in multi-threaded initialization
4. **Optional Dependencies**: Slow or hanging optional imports (like K3D)
5. **Symbolic Computation**: Heavy symbolic algebra initialization

POTENTIAL SOLUTIONS:

1. **Lazy Imports**: Move heavy imports inside functions/classes
2. **Import Guards**: Use try/except for optional imports
3. **Reduce Module-Level Code**: Move initialization to functions
4. **Break Circular Dependencies**: Reorganize imports
5. **Cache Symbolic Expressions**: Avoid recomputing on each import

Next steps:
- Check which specific test showed hanging behavior
- Review the import graph for cycles
- Consider lazy loading for heavy components
""")

# Show final module state
bmcs_modules = [name for name in sys.modules.keys() if name.startswith('bmcs_utils')]
print(f"\nCurrently loaded bmcs_utils modules: {len(bmcs_modules)}")
for module in sorted(bmcs_modules):
    print(f"  - {module}")

=== ANALYSIS AND RECOMMENDATIONS ===

Based on the tests above, common causes of hanging imports include:

1. **Circular Dependencies**: Check if modules import each other
2. **Heavy Initialization**: Long-running code in module-level scope
3. **Threading Issues**: Deadlocks in multi-threaded initialization
4. **Optional Dependencies**: Slow or hanging optional imports (like K3D)
5. **Symbolic Computation**: Heavy symbolic algebra initialization

POTENTIAL SOLUTIONS:

1. **Lazy Imports**: Move heavy imports inside functions/classes
2. **Import Guards**: Use try/except for optional imports
3. **Reduce Module-Level Code**: Move initialization to functions
4. **Break Circular Dependencies**: Reorganize imports
5. **Cache Symbolic Expressions**: Avoid recomputing on each import

Next steps:
- Check which specific test showed hanging behavior
- Review the import graph for cycles
- Consider lazy loading for heavy components


Currently loaded bmcs_utils modules: 45
  - bmcs_utils
  - bmcs_ut